# CRSP Daily Data Learning: Analysis Framework and Client Story

**Data coverage:** ../data/crsp_dsf_2024.csv.gz, 2024-01-02 through 2024-12-31.  
**Objective:** Convert the CRSP Daily Stock File (DSF) into reproducible documentation, quality checks, and a client-facing narrative.

> The local DSF extract does not include SHRCD or EXCHCD. To define common stocks on the three major U.S. exchanges, join a names/history table on both permno and its effective date; do not join on permno alone.

## 1. Project description for the client

Using CRSP security-level daily data, we create an auditable profile of equity liquidity and trading activity. We define a tradable universe using price, volume, returns, and shares outstanding, then create reproducible security features for TAQ sampling, market-microstructure research, or portfolio analysis.

**Suggested 30-second opening:**

> This project uses authoritative CRSP daily security data to establish a 2024 U.S. equity baseline panel. Instead of selecting only the most active stocks, we jointly control for price, sustained trading days, and liquidity while retaining cross-sectional coverage by market capitalization and trading activity. The resulting sample is economically tradable and remains representative within the storage budget when linked to higher-frequency TAQ data.

**Avoid overclaiming:** This file does not contain ticker or company name, and cannot independently identify common stocks or exchanges. Those determinations require a point-in-time names/history join.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path('../data/crsp_dsf_2024.csv.gz')
assert DATA.exists(), f'Data file not found: {DATA.resolve()}'

# PERMNO is a stable CRSP security-level identifier; CUSIP can change over time.
dtype = {'permno': 'Int64', 'permco': 'Int64', 'date': 'string'}
dsf = pd.read_csv(DATA, compression='gzip', dtype=dtype, low_memory=False)
dsf['date'] = pd.to_datetime(dsf['date'], errors='coerce')
dsf.shape, dsf['date'].min(), dsf['date'].max()

((2400962, 20),
 Timestamp('2024-01-02 00:00:00'),
 Timestamp('2024-12-31 00:00:00'))

In [5]:
dsf.columns.tolist()

['cusip',
 'permno',
 'permco',
 'issuno',
 'hexcd',
 'hsiccd',
 'date',
 'bidlo',
 'askhi',
 'prc',
 'vol',
 'ret',
 'bid',
 'ask',
 'shrout',
 'cfacpr',
 'cfacshr',
 'openprc',
 'numtrd',
 'retx']

## 2. Inspect a data slice

This compact, date-sorted sample presents the core security-day fields used in the analysis. Change SAMPLE_DATE or HEAD_ROWS to inspect another day or a larger slice.

In [2]:
SAMPLE_DATE = dsf['date'].min()
HEAD_ROWS = 10
SLICE_COLUMNS = ['date', 'permno', 'permco', 'cusip', 'prc', 'ret', 'retx', 'vol', 'shrout']

(dsf.loc[dsf['date'].eq(SAMPLE_DATE), SLICE_COLUMNS]
    .sort_values('permno')
    .head(HEAD_ROWS)
    .reset_index(drop=True))

,date,permno,permco,cusip,prc,ret,retx,vol,shrout
0,2024-01-02,10026,7976,46603210,168.86,0.010291,0.010291,89969.0,19367.0
1,2024-01-02,10028,7978,29402E10,4.71,-0.030864,-0.030864,18599.0,26700.0
2,2024-01-02,10032,7980,72913210,106.35,-0.016462,-0.016462,78780.0,27504.0
3,2024-01-02,10044,7992,77467X10,4.94,0.073913,0.073913,7736.0,6304.0
4,2024-01-02,10065,20023,00621210,17.45,-0.014681,-0.014681,235208.0,120810.0
5,2024-01-02,10066,6331,35518410,3.35,-0.011800,-0.011800,32753.0,11784.0
6,2024-01-02,10104,8045,68389X10,104.06,-0.012994,-0.012994,9597500.0,2748922.0
7,2024-01-02,10107,8048,59491810,370.87,-0.013749,-0.013749,25134559.0,7432262.0
8,2024-01-02,10113,53202,00768Y20,55.02,-0.021345,-0.021345,598.0,450.0
9,2024-01-02,10138,8087,74144T10,107.91,0.002043,0.002043,1606393.0,223938.0


## 3. Field dictionary and units

| Business concept | Field / definition | Interpretation and common mistake |
|---|---|---|
| Security, date | permno, date | The panel key should be (permno, date); do not treat CUSIP as a permanent key. |
| Closing price | abs(prc) | A negative CRSP prc commonly indicates a bid/ask-derived price. Use its absolute value; do not simply delete it. |
| Total return | ret | Daily return including available distributions such as dividends; the default choice for return research. |
| Ex-distribution return | retx | Use only when distributions should explicitly be excluded. |
| Volume | vol | Measured in shares. |
| Shares outstanding | shrout * 1,000 | shrout is reported in thousands. Omitting the multiplier misstates market cap and turnover by three orders of magnitude. |
| Dollar volume | abs(prc) * vol | A common scale for price and trading activity. |
| Turnover | vol / (shrout * 1,000) | Daily shares traded as a share of shares outstanding. |

In [ ]:
key_cols = ['permno', 'date']
print(f'Observations: {len(dsf):,}')
print(f'Securities (PERMNO): {dsf.permno.nunique():,}')
print(f'Trading days: {dsf.date.nunique():,}')
print(f'Duplicate panel keys: {dsf.duplicated(key_cols).sum():,}')

quality = pd.DataFrame({
    'missing_n': dsf.isna().sum(),
    'missing_pct': dsf.isna().mean().mul(100).round(3),
    'dtype': dsf.dtypes.astype(str)
}).sort_values('missing_pct', ascending=False)
quality

## 4. Reproducible derived variables and quality rules

Raw fields are retained and analysis variables are created separately. Missing, non-finite, or non-positive prices, volumes, and shares outstanding are excluded from the relevant metric rather than imputed as zero.

In [ ]:
x = dsf.copy()
x['price'] = x['prc'].abs()
x['shares_outstanding'] = x['shrout'] * 1_000
valid_price = np.isfinite(x['price']) & x['price'].gt(0)
valid_vol = np.isfinite(x['vol']) & x['vol'].gt(0)
valid_shares = np.isfinite(x['shares_outstanding']) & x['shares_outstanding'].gt(0)
x['dollar_volume'] = (x['price'] * x['vol']).where(valid_price & valid_vol)
x['market_cap'] = (x['price'] * x['shares_outstanding']).where(valid_price & valid_shares)
x['turnover'] = (x['vol'] / x['shares_outstanding']).where(valid_vol & valid_shares)

pd.Series({
    'negative CRSP prices (handled with abs)': x['prc'].lt(0).sum(),
    'invalid price': (~valid_price).sum(),
    'invalid volume': (~valid_vol).sum(),
    'invalid shares outstanding': (~valid_shares).sum(),
    'usable dollar-volume rows': x['dollar_volume'].notna().sum(),
})

## 5. Security-level profile: ready for screening and client reporting

The example first applies a daily price threshold of price >= $5, then calculates security-level statistics. For TAQ sampling, trading_days >= 200 and median dollar volume >= $5m are transparent initial screening rules. Fix and document the thresholds at project launch.

In [ ]:
usable = x.loc[x['price'].ge(5)].copy()
features = (usable.groupby('permno', as_index=False)
    .agg(
        trading_days=('date', 'nunique'),
        med_price=('price', 'median'),
        adv=('vol', 'mean'),
        med_dollar_volume=('dollar_volume', 'median'),
        adv_dollar=('dollar_volume', 'mean'),
        med_turnover=('turnover', 'median'),
        annual_volatility=('ret', lambda s: s.std() * np.sqrt(252)),
        med_market_cap=('market_cap', 'median')
    ))
eligible = features.query('trading_days >= 200 and med_dollar_volume >= 5_000_000').copy()
print(f'$5+ daily observations: {len(usable):,}; securities: {features.permno.nunique():,}')
print(f'Securities passing the example initial screen: {len(eligible):,}')
features.describe(percentiles=[.05, .25, .5, .75, .95]).T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(np.log10(eligible['med_market_cap'].dropna()), bins=40, color='#3973ac')
axes[0].set(title='Initially screened securities: market-cap distribution', xlabel='log10(median market cap, USD)', ylabel='Number of securities')
axes[1].hist(np.log10(eligible['adv_dollar'].dropna()), bins=40, color='#cf7a38')
axes[1].set(title='Initially screened securities: dollar ADV distribution', xlabel='log10(dollar ADV, USD)', ylabel='Number of securities')
plt.tight_layout()

## 6. Explaining the data and method to the client

**Data source and granularity.** CRSP DSF is historical market data at the security-trading-day level. It provides daily price, total return, volume, and shares outstanding, while stable PERMNO identifiers manage security history. The current extract covers 252 trading days in 2024; the checks above report its actual security and observation counts.

**Why these measures fit the project.** A price threshold helps exclude economically difficult-to-trade low-price states. Dollar volume reflects price and trading activity; turnover normalizes volume by firm size; and total return measures the daily return received by an investor. Together they avoid treating one metric as liquidity.

**Method discipline.** Apply universe conditions using daily information; assign classifications through effective-date joins; and aggregate features only over observations that pass daily conditions. Each step reports sample counts, missingness, thresholds, and PERMNO lists, so a client can audit inclusion and exclusion.

**Known boundaries.** Daily CRSP supports universe definition and low-frequency features, but does not answer order-book, trade-by-trade, or NBBO questions; those require TAQ after screening.

## 7. Suggested client-meeting structure

1. **Objective:** The business or research question, and why it requires a balanced, tradable sample.
2. **Data:** CRSP DSF security-day granularity, time coverage, key fields, and units.
3. **Rules:** Point-in-time security classification, absolute price, the $5 daily threshold, trading-day and dollar-volume thresholds.
4. **Deliverables:** A reproducible feature table, screening audit table, cross-sectional overview, and subsequent TAQ sample.
5. **Risks / assumptions:** CRSP-to-TAQ linkage coverage, corporate actions, missing values, and pilot calibration of the TAQ storage budget.

**Confirm before the meeting:** whether the client needs total return (ret) or ex-distribution return (retx); whether the universe is limited to common stocks and NYSE-Nasdaq-AMEX; and the TAQ storage limit and representativeness requirements.

## 8. Next steps

- Obtain dsenames (or an equivalent names/history table), then perform a point-in-time join using permno and date between namedt and nameendt; apply shrcd in {10,11} and exchcd in {1,2,3}.
- Record sample size and exclusion reasons at every screening stage, and export features as a versioned project intermediate.
- For TAQ use, first download a pilot spanning market-cap and liquidity buckets, then use observed compressed sizes to calibrate the download budget; do not treat CRSP activity proxies as direct GB forecasts.